# Attract-Repel Link Prediction on Cora

Link Prediction on Cora (Planetoid): Attract-Repel embeddings split a node's latent vector into an "attract" half and a "repel" half, scoring an edge as `attract_similarity - repel_similarity` so the model can represent both homophily and heterophily. This notebook ports the original PyTorch Geometric reference script to K3-Node: a `K3GCNEncoder` (`GCNConv` x2) produces node embeddings, held-out positive/negative edges are scored by either `K3ARLinkPredictor` (Attract-Repel, used by default) or `K3LinkPredictor` (a traditional concatenation-MLP decoder), and the model is trained for 200 epochs on a hand-rolled train/val/test edge split with per-epoch negative sampling, reporting validation/test ROC-AUC and the Attract-Repel "R-fraction". The single code cell below installs **K3-Node** and runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable — including a genuine stateless JAX training step.

In [ ]:
# Setup environment and install dependencies
!pip install --upgrade git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
# Switch to your preferred backend: 'torch', 'tensorflow', or 'jax'
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import numpy as np
import keras
from keras import layers, ops

from k3_node.datasets import Planetoid
from k3_node.layers import GCNConv
from k3_node.models.utils import negative_sampling

title = "Attract-Repel Link Prediction on Cora"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset & a hand-rolled train/val/test edge split (parity with the
# deprecated `torch_geometric.utils.train_test_split_edges`)
dataset = Planetoid(root="./data/Planetoid", name="Cora")
data = dataset[0]


def train_test_split_edges(data, val_ratio=0.05, test_ratio=0.1):
    edge_index = ops.convert_to_numpy(data.edge_index)
    num_nodes = int(np.max(edge_index)) + 1

    mask = edge_index[0] < edge_index[1]
    edges = edge_index[:, mask]
    num_edges = edges.shape[1]

    perm = np.random.permutation(num_edges)
    n_val = int(val_ratio * num_edges)
    n_test = int(test_ratio * num_edges)

    val_edges = edges[:, perm[:n_val]]
    test_edges = edges[:, perm[n_val:n_val + n_test]]
    train_edges = edges[:, perm[n_val + n_test:]]

    def make_undirected(e):
        return np.concatenate([e, e[[1, 0]]], axis=1)

    data.train_pos_edge_index = ops.convert_to_tensor(make_undirected(train_edges), dtype="int64")
    data.val_pos_edge_index = ops.convert_to_tensor(val_edges, dtype="int64")
    data.test_pos_edge_index = ops.convert_to_tensor(test_edges, dtype="int64")
    data.val_neg_edge_index = negative_sampling(data.edge_index, num_nodes=num_nodes, num_neg_samples=val_edges.shape[1])
    data.test_neg_edge_index = negative_sampling(data.edge_index, num_nodes=num_nodes, num_neg_samples=test_edges.shape[1])
    return data


data = train_test_split_edges(data)
num_features = dataset.num_features
hidden_channels = 128
out_channels = 64
use_ar = True  # set to False to use the traditional K3LinkPredictor decoder instead


# 2. GCN encoder shared by both predictor heads
class K3GCNEncoder(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def call(self, x, edge_index):
        x = ops.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)


# 3. Traditional decoder: an MLP over the concatenation of both endpoints
class K3LinkPredictor(keras.Model):
    def __init__(self, hidden_channels):
        super().__init__()
        self.lin1 = layers.Dense(hidden_channels)
        self.lin2 = layers.Dense(1)

    def call(self, z_i, z_j):
        x = ops.relu(self.lin1(ops.concatenate([z_i, z_j], axis=1)))
        return ops.reshape(self.lin2(x), (-1,))


# 4. Attract-Repel decoder: splits the embedding into an "attract" half and a
# "repel" half, scoring an edge as attract_similarity - repel_similarity
class K3ARLinkPredictor(keras.layers.Layer):
    def __init__(self, in_channels):
        super().__init__()
        self.attract_dim = in_channels // 2

    def call(self, z_i, z_j):
        z_i_attr, z_i_repel = z_i[:, :self.attract_dim], z_i[:, self.attract_dim:]
        z_j_attr, z_j_repel = z_j[:, :self.attract_dim], z_j[:, self.attract_dim:]
        attract_score = ops.sum(z_i_attr * z_j_attr, axis=1)
        repel_score = ops.sum(z_i_repel * z_j_repel, axis=1)
        return attract_score - repel_score


encoder = K3GCNEncoder(num_features, hidden_channels, out_channels)
predictor = K3ARLinkPredictor(out_channels) if use_ar else K3LinkPredictor(hidden_channels)
mode = "Attract-Repel embeddings" if use_ar else "Traditional embeddings"
print(f"Running link prediction on Cora with {mode}")

# Eager forward pass to build weights with backend tensors
z0 = encoder(data.x, data.train_pos_edge_index)
_ = predictor(z0[:2], z0[:2])

bce_loss = keras.losses.BinaryCrossentropy(from_logits=True)
optimizer = keras.optimizers.Adam(learning_rate=0.01)

# Setup optimizer variables for the functional (JAX) backend
if backend == "jax":
    import jax
    import jax.numpy as jnp

    trainable_vars = encoder.trainable_variables + predictor.trainable_variables
    optimizer.build(trainable_vars)
    opt_vars = [v.value for v in optimizer.variables]
    enc_non_trainable = [v.value for v in encoder.non_trainable_variables]
    pred_non_trainable = [v.value for v in predictor.non_trainable_variables]


# 5. Multi-Backend Training Step
def sample_neg_edges():
    return negative_sampling(
        data.train_pos_edge_index,
        num_nodes=data.num_nodes,
        num_neg_samples=int(ops.shape(data.train_pos_edge_index)[1]),
    )


def train():
    global opt_vars
    trainable_vars = encoder.trainable_variables + predictor.trainable_variables
    neg_edge_index = sample_neg_edges()

    if backend == "torch":
        z = encoder(data.x, data.train_pos_edge_index)
        pos_out = predictor(ops.take(z, data.train_pos_edge_index[0], axis=0), ops.take(z, data.train_pos_edge_index[1], axis=0))
        neg_out = predictor(ops.take(z, neg_edge_index[0], axis=0), ops.take(z, neg_edge_index[1], axis=0))
        loss = bce_loss(ops.ones_like(pos_out), pos_out) + bce_loss(ops.zeros_like(neg_out), neg_out)
        loss.backward()
        grads = [v.value.grad for v in trainable_vars]
        optimizer.apply_gradients(zip(grads, trainable_vars))
        for v in trainable_vars:
            if v.value.grad is not None:
                v.value.grad.zero_()
        return float(ops.convert_to_numpy(loss))

    elif backend == "tensorflow":
        import tensorflow as tf
        with tf.GradientTape() as tape:
            z = encoder(data.x, data.train_pos_edge_index)
            pos_out = predictor(ops.take(z, data.train_pos_edge_index[0], axis=0), ops.take(z, data.train_pos_edge_index[1], axis=0))
            neg_out = predictor(ops.take(z, neg_edge_index[0], axis=0), ops.take(z, neg_edge_index[1], axis=0))
            loss = bce_loss(ops.ones_like(pos_out), pos_out) + bce_loss(ops.zeros_like(neg_out), neg_out)
        grads = tape.gradient(loss, trainable_vars)
        optimizer.apply_gradients(zip(grads, trainable_vars))
        return float(ops.convert_to_numpy(loss))

    else:  # jax
        enc_trainable = [v.value for v in encoder.trainable_variables]
        pred_trainable = [v.value for v in predictor.trainable_variables]

        def loss_fn(e_vars, p_vars):
            z, _ = encoder.stateless_call(e_vars, enc_non_trainable, data.x, data.train_pos_edge_index)
            pos_i = ops.take(z, data.train_pos_edge_index[0], axis=0)
            pos_j = ops.take(z, data.train_pos_edge_index[1], axis=0)
            neg_i = ops.take(z, neg_edge_index[0], axis=0)
            neg_j = ops.take(z, neg_edge_index[1], axis=0)
            pos_out, _ = predictor.stateless_call(p_vars, pred_non_trainable, pos_i, pos_j)
            neg_out, _ = predictor.stateless_call(p_vars, pred_non_trainable, neg_i, neg_j)
            pos_loss = -jnp.mean(jax.nn.log_sigmoid(pos_out))
            neg_loss = -jnp.mean(jax.nn.log_sigmoid(-neg_out))
            return pos_loss + neg_loss

        loss_val, (enc_grads, pred_grads) = jax.value_and_grad(loss_fn, argnums=(0, 1))(enc_trainable, pred_trainable)

        new_trainable, opt_vars = optimizer.stateless_apply(
            opt_vars, enc_grads + pred_grads, enc_trainable + pred_trainable
        )
        for v, val in zip(trainable_vars, new_trainable):
            v.assign(val)
        return float(loss_val)


def compute_auc(pos_out, neg_out):
    from sklearn.metrics import roc_auc_score
    y = np.concatenate([np.ones(ops.shape(pos_out)[0]), np.zeros(ops.shape(neg_out)[0])])
    scores = np.concatenate([
        ops.convert_to_numpy(ops.sigmoid(pos_out)),
        ops.convert_to_numpy(ops.sigmoid(neg_out)),
    ])
    return roc_auc_score(y, scores)


def evaluate():
    z = encoder(data.x, data.train_pos_edge_index)

    def score(edge_index):
        return predictor(ops.take(z, edge_index[0], axis=0), ops.take(z, edge_index[1], axis=0))

    val_auc = compute_auc(score(data.val_pos_edge_index), score(data.val_neg_edge_index))
    test_auc = compute_auc(score(data.test_pos_edge_index), score(data.test_neg_edge_index))
    return val_auc, test_auc


print(f"Training K3-Node link predictor on {backend} backend...")
best_val_auc, final_test_auc = 0.0, 0.0
for epoch in range(1, 201):
    loss = train()
    val_auc, test_auc = evaluate()
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        final_test_auc = test_auc
    if epoch % 10 == 0:
        print(f"Epoch: {epoch:03d}, Loss: {loss:.4f}, Val AUC: {val_auc:.4f}, Test AUC: {test_auc:.4f}")

print(f"Final results - Val AUC: {best_val_auc:.4f}, Test AUC: {final_test_auc:.4f}")

# R-fraction: share of embedding energy living in the repel subspace
if use_ar:
    z = encoder(data.x, data.train_pos_edge_index)
    attract_dim = out_channels // 2
    attract_norm_sq = ops.sum(ops.square(z[:, :attract_dim]))
    repel_norm_sq = ops.sum(ops.square(z[:, attract_dim:]))
    r_fraction = repel_norm_sq / (attract_norm_sq + repel_norm_sq)
    print(f"R-fraction: {float(ops.convert_to_numpy(r_fraction)):.4f}")

print("\n✓ K3-Node execution completed successfully!")